In [ ]:
import os
import pandas as pd
import warnings
from src.config import *

In [ ]:
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

In [ ]:
region_list = ['강릉', '대관령']

In [ ]:
"""
지역별·연도별로 다운로드한 받은 종관기상관측(asos) 데이터를 지역기준으로 하나로 합쳐 asos_{region}_2002_2025.csv 파일 형태로 저장

[입력]
- region_list: 고랭지배추 주 산지 정보가 담긴 리스트
- 파일경로: raw_data/asos/input/SURFACE_ASOS_{지역코드}_DAY_2002 ~ 2025.zip

[경로 구성]
- 현재 경로와 파일명을 기반으로 file_paths 리스트에 현재 경로 + 파일명을 더한 전체 경로를 저장

[1차 가공]
- 지역별로 asos 데이터를 하나의 데이터프레임으로 합치기 위해 concat_df 생성
- 각 asos 파일별로 read_csv 함수를 통해 데이터프레임으로 변환
- 합계 일사량이라는 컬럼명이 파일마다 "합계 일사량" 혹은 "합계 일사"로 네이밍이 되어있어 통합
- 59개 컬럼 중 일시를 포함한 11개의 컬럼만 추출한 데이터프레임(select_df)을 pd.concat([concat_df, select_df]) 사용

[2차 가공]
- 일시 데이터를 기반으로 시간순으로 정렬
- 원천 데이터(일강수량 컬럼)에서 비가 오지 않은 날이 NaN으로 기록되어 있어, fillna(0)으로 0 처리

[3차 가공]
- 일시와 일강수량 컬럼을 제외한 나머지 컬럼들은 결측치를 위해 선형 보간법을 사용
- interpolate(linear) -> bfill -> ffill 총 3단계 적용

[출력]
- raw_data/asos/output/asos_{region}_2002_2025.csv
"""
for n, region in enumerate(region_list):
    # 현재 경로 확인
    data_folder = RAW_DATA_DIR + 'asos/input/' + region +'/'
    print(data_folder)

    # 파일명 확인
    # os.listdir()은 파일의 순서를 보장하지 않아서 concat 작업 이후에 시간순으로 정렬하는 작업이 필요
    file_names = os.listdir(data_folder)
    print(file_names[:1])

    # 데이터 폴더 속 파일 각각 경로 확인
    file_paths = []
    for file_name in file_names:
        file_paths.append(os.path.join(data_folder, file_name))
        print(file_name)

    # csv 파일들을 DataFrame으로 불러와서 concat
    concat_df = pd.DataFrame()
    for file_path in file_paths:
        df = pd.read_csv(file_path, encoding='cp949')
        

        # 특정 파일에는 합계 일사량이 아니라 합계 일사로 작성되어있는 컬럼이 존재하여 이를 rename하는 작업 필요
        if "합계 일사(MJ/m2)" in df.columns:
            df = df.rename(columns={"합계 일사(MJ/m2)": "합계 일사량(MJ/m2)"})
    
        # 59개의 컬럼 중 10개만 추출
        selected_df = df[[
            '일시', 
            '평균기온(°C)', 
            '최저기온(°C)', 
            '최고기온(°C)', 
            '일강수량(mm)',
            '평균 이슬점온도(°C)', 
            '최소 상대습도(%)', 
            '평균 상대습도(%)',
            '합계 일사량(MJ/m2)', 
            '평균 지면온도(°C)', 
            '최저 초상온도(°C)'
        ]]

        concat_df = pd.concat([concat_df, selected_df])

    concat_df['일시'] = pd.to_datetime(concat_df['일시'])
    concat_df = concat_df.sort_values('일시').reset_index(drop=True) # os.listdir()은 파일의 순서를 보장하지 않기 때문에 정렬 필요
    concat_df['일강수량(mm)'] = concat_df['일강수량(mm)'].fillna(0) # 비가 안 온날이 NaN으로 기록되어서, 이동평균 계산 전 fillna(0)으로 처리

    # 보간법
    numeric_cols = [c for c in concat_df.columns if c not in ['일시', '일강수량(mm)']] # 일강수량은 별도로 위에서 처리했고, 일시는 날짜 타입이므로 제외

    # as-is
    # concat_df['합계 일사량(MJ/m2)'] = concat_df['합계 일사량(MJ/m2)'].interpolate(method='linear').fillna(method='bfill').fillna(method='ffill')
    # concat_df['평균 지면온도(°C)'] = concat_df['평균 지면온도(°C)'].interpolate(method='linear').fillna(method='bfill').fillna(method='ffill')
    # concat_df['최저 초상온도(°C)'] = concat_df['최저 초상온도(°C)'].interpolate(method='linear').fillna(method='bfill').fillna(method='ffill')

    # TODO: 이런 표현 잘 알아두자 (to-be)
    concat_df[numeric_cols] = (
        concat_df[numeric_cols] # 지정된 열의 데이터를 꺼내서
        .interpolate(method='linear') # 선형 보간법
        .bfill() # 데이터의 시작 부분에서 interpolate가 처리 못하는 경우를 위한 안전장치 
        .ffill() # 데이터의 끝 부분에서 interpolate가 처리 못하는 경우를 위한 안전장치
    )

    # csv export 전 NaN 값 확인
    nan_column_count = concat_df.isnull().sum()
    print("[load_asos_nan]\n", nan_column_count)

    # 지역별로 합쳐진 하나의 DataFrame을 csv로 export
    concat_df.to_csv(PROCESSED_DIR + 'asos_' + region_list[n] + '_2002_2025.csv', encoding='utf-8-sig', index=False)
    